In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests
import io
import json
import os

API_BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:8080")
DATA_ENDPOINT = f"{API_BASE_URL}/data"
ui_state = {"data_id": None}

asset_upload = widgets.FileUpload(
    accept='.csv, .json',
    multiple=False,
    description='Asset Returns:'
)

benchmark_upload = widgets.FileUpload(
    accept='.csv, .json',
    multiple=False,
    description='Benchmark (Optional):'
)

upload_button = widgets.Button(
    description='Upload Data',
    button_style='info',
    tooltip='Upload selected files to the PorQua API',
    icon='upload'
)

output_area = widgets.Output()
# ----------------------

print("Widgets created.")

Widgets created.


In [2]:
def on_upload_button_clicked(b):
    """Handles the logic when the upload button is clicked."""
    print(f"DEBUG: Inside button click. asset_upload.value is: {asset_upload.value}")
    output_area.clear_output()

    asset_files = asset_upload.value
    benchmark_files = benchmark_upload.value

    if not asset_files:
        with output_area:
            print("Error: Asset Returns file is required.")
        return

    # request 
    # ipywidgets FileUpload value is a tuple even for single file: ({'name': 'file.csv', 'type': 'text/csv', ... 'content': b'...'},)
    asset_info = asset_files[0] # Get the dict for the single asset file
    files_to_send = {
        'asset_file': (asset_info['name'], io.BytesIO(asset_info['content']), asset_info['type'])
    }

    if benchmark_files:
        benchmark_info = benchmark_files[0]
        files_to_send['benchmark_file'] = (benchmark_info['name'], io.BytesIO(benchmark_info['content']), benchmark_info['type'])
        print(f"Preparing to upload: Assets='{asset_info['name']}', Benchmark='{benchmark_info['name']}'") # Debug
    else:
            print(f"Preparing to upload: Assets='{asset_info['name']}', No Benchmark") # Debug
    
    with output_area: # Display messages within the output widget
        print("Uploading...")
        try:
            response = requests.post(DATA_ENDPOINT, files=files_to_send)
            response.raise_for_status()
            result = response.json()
            ui_state['data_id'] = result.get('data_id')

            print("\n--- Upload Successful! ---")
            print(f"Data ID: {result.get('data_id')}")
            print(f"Assets Found: {result.get('num_assets')} ({result.get('assets', [])[:5]}...)")
            print(f"Time Periods: {result.get('num_dates')}")
            if result.get('has_benchmark'):
                print(f"Benchmark Included: Yes ('{result.get('benchmark_name')}')")
            else:
                print("Benchmark Included: No")
            print("-------------------------")
            asset_upload.value = ()
            benchmark_upload.value = ()


        except requests.exceptions.ConnectionError:
                print(f"\nError: Could not connect to the API at {API_BASE_URL}. Is the server running?")
        except requests.exceptions.RequestException as e:
                print(f"\n--- Upload Failed! ---")
                print(f"Error Code: {response.status_code}")
                try:
                    error_detail = response.json().get('detail', 'No detail provided.')
                    print(f"API Error: {error_detail}")
                except json.JSONDecodeError:
                    print(f"API Error: Could not decode error response. Raw response:\n{response.text}")
                except AttributeError:
                    print(f"Error sending request: {e}")
                print("-----------------------")
        except Exception as e:
                print(f"\n--- An unexpected error occurred ---")
                print(str(e))
                print("------------------------------------")

print("Upload function defined.")

Upload function defined.


In [3]:
upload_button.on_click(on_upload_button_clicked)
print("Button click handler attached.")

Button click handler attached.


In [4]:
ui_box = widgets.VBox([
    widgets.HTML("<h3>Upload Data</h3>"),
    asset_upload,
    benchmark_upload,
    upload_button,
    output_area
])

display(ui_box)